In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# Household composition
# To download original dataset go to - 
# https://www.ons.gov.uk/datasets/TS021/editions/2021/versions/3
household_composition_csv = r"N:\Geodatabase\Raw_Data\Census 2021\Household composition and Size\TS003 - Household composition.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframe
lsoa_household_composition_df_P = pd.read_csv(household_composition_csv, skiprows = 7,nrows = 35672, skip_blank_lines = True)
lsoa_household_composition_df_P.head()

,2021 super output area - lower layer,Total: All households,One-person household,One-person household: Aged 66 years and over,One-person household: Other,Single family household,Single family household: All aged 66 years and over,Single family household: Married or civil partnership couple,Single family household: Married or civil partnership couple: No children,Single family household: Married or civil partnership couple: Dependent children,Single family household: Married or civil partnership couple: All children non-dependent,Single family household: Cohabiting couple family,Single family household: Cohabiting couple family: No children,Single family household: Cohabiting couple family: With dependent children,Single family household: Cohabiting couple family: All children non-dependent,Single family household: Lone parent family,Single family household: Lone parent family: With dependent children,Single family household: Lone parent family: All children non-dependent,Single family household: Other single family household,Single family household: Other single family household: Other family composition,Other household types,Other household types: With dependent children,"Other household types: Other, including all full-time students and all aged 66 years and over"
0,E01011954 : Hartlepool 001A,965,266,103,163,671,69,260,100,100,60,152,50,89,13,188,135,53,2,2,28,13,15
1,E01011969 : Hartlepool 001B,599,157,92,65,431,116,189,78,61,50,70,27,36,7,56,33,23,0,0,11,7,4
2,E01011970 : Hartlepool 001C,488,122,65,57,349,83,174,79,44,51,50,24,23,3,39,25,14,3,3,17,9,8
3,E01011971 : Hartlepool 001D,521,104,32,72,406,42,235,77,102,56,68,33,30,5,61,37,24,0,0,11,6,5
4,E01033465 : Hartlepool 001F,741,151,42,109,566,46,317,87,175,55,126,59,63,4,73,59,14,4,4,24,9,15


In [6]:
#drop columns
lsoa_household_composition_df_P.drop(['Single family household: Other single family household: Other family composition'], 1, inplace=True)

# Dictionary for renaming columns with corrected keys
column_rename_map = {
    "Total: All households": "total_household_composition_pop",
    "One-person household": "one_person",
    "One-person household: Aged 66 years and over": "one_person_aged_66_years_and_over",
    "One-person household: Other": "one_person_other",
    "Single family household": "single_family",
    "Single family household: All aged 66 years and over": "single_family_all_aged_66_years_and_over",
    "Single family household: Married or civil partnership couple": "single_family_married_or_civil_partnership",
    "Single family household: Married or civil partnership couple: No children": "single_family_married_or_civil_partnership_no_children",
    "Single family household: Married or civil partnership couple: Dependent children": "single_family_married_or_civil_partnership_dependant_children",
    "Single family household: Married or civil partnership couple: All children non-dependent": "single_family_married_or_civil_partnership_nondep_children",
    "Single family household: Cohabiting couple family": "single_family_cohabiting",
    "Single family household: Cohabiting couple family: No children": "single_family_cohabiting_no_children",
    "Single family household: Cohabiting couple family: With dependent children": "single_family_cohabiting_dependant_children",
    "Single family household: Cohabiting couple family: All children non-dependent": "single_family_cohabiting_nondep_children",
    "Single family household: Lone parent family": "single_family_lone_parent",
    "Single family household: Lone parent family: All children non-dependent": "single_family_lone_parent_nondep_children",
    "Single family household: Lone parent family: With dependent children": "single_family_lone_parent_dependant_children",
    "Single family household: Other single family household": "single_family_other",
    "Other household types": "other_household_types",    
    "Other household types: With dependent children": "other_types_dependant_children",
    "Other household types: Other, including all full-time students and all aged 66 years and over": "other_types_all_ft_students_and_all_aged_66_years_over"
}    
       
# Rename columns using the dictionary
lsoa_household_composition_df_P.rename(columns=column_rename_map, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_household_composition_df_P.columns = (
    lsoa_household_composition_df_P.columns
    .str.cat(['_count'] * len(lsoa_household_composition_df_P.columns))  # Append '_count' to each column
)

#split 2021 super output area - lower layer_count
# Step 1: Split the original column
lsoa_household_composition_df_P[['lsoa21cd', 'lsoa21nm']] = (
    lsoa_household_composition_df_P['2021 super output area - lower layer_count']
    .str.split(':', expand=True)
)

# Step 2: Strip whitespace
lsoa_household_composition_df_P['lsoa21cd'] = lsoa_household_composition_df_P['lsoa21cd'].str.strip()
lsoa_household_composition_df_P['lsoa21nm'] = lsoa_household_composition_df_P['lsoa21nm'].str.strip()

# Step 3: Drop the original column
lsoa_household_composition_df_P.drop(columns=['2021 super output area - lower layer_count'], inplace=True)

# Step 4: Move the two new columns to the front
cols = ['lsoa21cd', 'lsoa21nm'] + [col for col in lsoa_household_composition_df_P.columns if col not in ['lsoa21cd', 'lsoa21nm']]
lsoa_household_composition_df_P = lsoa_household_composition_df_P[cols]

lsoa_household_composition_df_P.drop(['lsoa21nm'], 1, inplace=True)

lsoa_household_composition_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\1850088287.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_household_composition_df_P.drop(['Single family household: Other single family household: Other family composition'], 1, inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\1850088287.py:56: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_household_composition_df_P.drop(['lsoa21nm'], 1, inplace=True)


,lsoa21cd,total_household_composition_pop_count,one_person_count,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_household_types_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count
0,E01011954,965,266,103,163,671,69,260,100,100,60,152,50,89,13,188,135,53,2,28,13,15
1,E01011969,599,157,92,65,431,116,189,78,61,50,70,27,36,7,56,33,23,0,11,7,4
2,E01011970,488,122,65,57,349,83,174,79,44,51,50,24,23,3,39,25,14,3,17,9,8
3,E01011971,521,104,32,72,406,42,235,77,102,56,68,33,30,5,61,37,24,0,11,6,5
4,E01033465,741,151,42,109,566,46,317,87,175,55,126,59,63,4,73,59,14,4,24,9,15


In [7]:
# Feature Engineering - Aggregating Ethnicity Counts
household_composition_groups = {
    "households_with_dependant_children_count": [
        "single_family_married_or_civil_partnership_dependant_children_count",
        "single_family_cohabiting_dependant_children_count",
        "single_family_lone_parent_dependant_children_count",
        "other_types_dependant_children_count"        
    ],
    "households_with_nondep_children_count": [
        "single_family_married_or_civil_partnership_nondep_children_count",
        "single_family_cohabiting_nondep_children_count",
        "single_family_lone_parent_nondep_children_count"
    ],
    "households_with_no_children_count": [
        "one_person_aged_66_years_and_over_count",
        "one_person_other_count",
        "single_family_all_aged_66_years_and_over_count",
        "single_family_married_or_civil_partnership_no_children_count",
        "single_family_cohabiting_no_children_count",
        "single_family_other_count",
        "other_types_all_ft_students_and_all_aged_66_years_over_count"        
    ]
}

# Compute aggregated ethnicity counts
for new_col, cols in household_composition_groups.items():
    lsoa_household_composition_df_P[new_col] = lsoa_household_composition_df_P[cols].sum(axis=1)

# Display results
lsoa_household_composition_df_P.head()

,lsoa21cd,total_household_composition_pop_count,one_person_count,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_household_types_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count
0,E01011954,965,266,103,163,671,69,260,100,100,60,152,50,89,13,188,135,53,2,28,13,15,337,126,502
1,E01011969,599,157,92,65,431,116,189,78,61,50,70,27,36,7,56,33,23,0,11,7,4,137,80,382
2,E01011970,488,122,65,57,349,83,174,79,44,51,50,24,23,3,39,25,14,3,17,9,8,101,68,319
3,E01011971,521,104,32,72,406,42,235,77,102,56,68,33,30,5,61,37,24,0,11,6,5,175,85,261
4,E01033465,741,151,42,109,566,46,317,87,175,55,126,59,63,4,73,59,14,4,24,9,15,306,73,362


## 2. Feature Engineering

In [8]:
# Create percentage columns for detailed ethnicity
for col in lsoa_household_composition_df_P.columns[2:]:
    perc_col = col.replace('_count', '_perc')
    lsoa_household_composition_df_P[perc_col] = (lsoa_household_composition_df_P[col] / lsoa_household_composition_df_P['total_household_composition_pop_count']) * 100

#lsoa_household_composition_df_P.drop(['total_household_composition_pop_count'], 1, inplace=True)
lsoa_household_composition_df_P.head()

,lsoa21cd,total_household_composition_pop_count,one_person_count,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_household_types_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,one_person_perc,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_household_types_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc
0,E01011954,965,266,103,163,671,69,260,100,100,60,152,50,89,13,188,135,53,2,28,13,15,337,126,502,27.564767,10.673575,16.891192,69.533679,7.150259,26.943005,10.362694,10.362694,6.217617,15.751295,5.181347,9.222798,1.347150,19.481865,13.989637,5.492228,0.207254,2.901554,1.347150,1.554404,34.922280,13.056995,52.020725
1,E01011969,599,157,92,65,431,116,189,78,61,50,70,27,36,7,56,33,23,0,11,7,4,137,80,382,26.210351,15.358932,10.851419,71.953255,19.365609,31.552588,13.021703,10.183639,8.347245,11.686144,4.507513,6.010017,1.168614,9.348915,5.509182,3.839733,0.000000,1.836394,1.168614,0.667780,22.871452,13.355593,63.772955
2,E01011970,488,122,65,57,349,83,174,79,44,51,50,24,23,3,39,25,14,3,17,9,8,101,68,319,25.000000,13.319672,11.680328,71.516393,17.008197,35.655738,16.188525,9.016393,10.450820,10.245902,4.918033,4.713115,0.614754,7.991803,5.122951,2.868852,0.614754,3.483607,1.844262,1.639344,20.696721,13.934426,65.368852
3,E01011971,521,104,32,72,406,42,235,77,102,56,68,33,30,5,61,37,24,0,11,6,5,175,85,261,19.961612,6.142035,13.819578,77.927063,8.061420,45.105566,14.779271,19.577735,10.748560,13.051823,6.333973,5.758157,0.959693,11.708253,7.101727,4.606526,0.000000,2.111324,1.151631,0.959693,33.589251,16.314779,50.095969
4,E01033465,741,151,42,109,566,46,317,87,175,55,126,59,63,4,73,59,14,4,24,9,15,306,73,362,20.377868,5.668016,14.709852,76.383266,6.207827,42.780027,11.740891,23.616734,7.422402,17.004049,7.962213,8.502024,0.539811,9.851552,7.962213,1.889339,0.539811,3.238866,1.214575,2.024291,41.295547,9.851552,48.852901


## 4. Load GIS shapefile 

In [9]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [10]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [11]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [12]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [13]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [14]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [15]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [16]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [17]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [18]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [19]:
census2021_household_composition_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_household_composition_df_P, how = 'left', on='lsoa21cd')
census2021_household_composition_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_household_composition_pop_count,one_person_count,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_household_types_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,one_person_perc,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_household_types_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,841,373,129,244,417,82,192,111,68,13,112,99,13,0,17,10,7,14,51,1,50,92,20,729,44.351962,15.338882,29.013080,49.583829,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,6.064209,0.118906,5.945303,10.939358,2.378121,86.682521
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,824,409,101,308,369,57,162,104,46,12,107,93,11,3,25,13,12,18,46,3,43,73,27,724,49.635922,12.257282,37.378641,44.781553,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,5.582524,0.364078,5.218447,8.859223,3.276699,87.864078
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1017,582,160,422,379,31,165,106,42,17,122,105,14,3,44,23,21,17,56,1,55,80,41,896,57.227139,15.732547,41.494592,37.266470,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.294985,4.326450,2.261554,2.064897,1.671583,5.506391,0.098328,5.408063,7.866273,4.031465,88.102262
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken

# 8. Final Tweaks

In [20]:
# Reorder columns in the combined dataframe

final_column_order = ['FID', 'lsoa21cd', 'lsoa21nm', 'geometry', 'msoa21cd', 'msoa21nm',
       'lad22cd', 'lad22nm', 'wd22cd', 'wd22nm', 'rgn22cd', 'rgn22nm',
       'data_source', 'data_resolution', 'data_time_period', 'data_web_link',
       'area_ha', 'total_household_composition_pop_count', 
       'one_person_count',
       'single_family_count', 
       'other_household_types_count',  
       'one_person_perc',
       'single_family_perc',  
       'other_household_types_perc',                 
       'one_person_aged_66_years_and_over_count', 
       'one_person_other_count',
       'single_family_all_aged_66_years_and_over_count',
       'single_family_married_or_civil_partnership_count',
       'single_family_married_or_civil_partnership_no_children_count',
       'single_family_married_or_civil_partnership_dependant_children_count',
       'single_family_married_or_civil_partnership_nondep_children_count',
       'single_family_cohabiting_count',
       'single_family_cohabiting_no_children_count',
       'single_family_cohabiting_dependant_children_count',
       'single_family_cohabiting_nondep_children_count',
       'single_family_lone_parent_count',
       'single_family_lone_parent_dependant_children_count',
       'single_family_lone_parent_nondep_children_count',
       'single_family_other_count',       
       'other_types_dependant_children_count',
       'other_types_all_ft_students_and_all_aged_66_years_over_count',
       'one_person_aged_66_years_and_over_perc', 
       'one_person_other_perc',       
       'single_family_all_aged_66_years_and_over_perc',
       'single_family_married_or_civil_partnership_perc',
       'single_family_married_or_civil_partnership_no_children_perc',
       'single_family_married_or_civil_partnership_dependant_children_perc',
       'single_family_married_or_civil_partnership_nondep_children_perc',
       'single_family_cohabiting_perc',
       'single_family_cohabiting_no_children_perc',
       'single_family_cohabiting_dependant_children_perc',
       'single_family_cohabiting_nondep_children_perc',
       'single_family_lone_parent_perc',
       'single_family_lone_parent_dependant_children_perc',
       'single_family_lone_parent_nondep_children_perc',
       'single_family_other_perc',        
       'other_types_dependant_children_perc',
       'other_types_all_ft_students_and_all_aged_66_years_over_perc',
       'households_with_dependant_children_count',
       'households_with_nondep_children_count',
       'households_with_no_children_count', 
       'households_with_dependant_children_perc',
       'households_with_nondep_children_perc',
       'households_with_no_children_perc'
]

census2021_household_composition_lsoa2021_gdb_df = census2021_household_composition_lsoa2021_gdb_df[final_column_order]

census2021_household_composition_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_household_composition_pop_count,one_person_count,single_family_count,other_household_types_count,one_person_perc,single_family_perc,other_household_types_perc,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,841,373,417,51,44.351962,49.583829,6.064209,129,244,82,192,111,68,13,112,99,13,0,17,10,7,14,1,50,15.338882,29.013080,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,0.118906,5.945303,92,20,729,10.939358,2.378121,86.682521
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,824,409,369,46,49.635922,44.781553,5.582524,101,308,57,162,104,46,12,107,93,11,3,25,13,12,18,3,43,12.257282,37.378641,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,0.364078,5.218447,73,27,724,8.859223,3.276699,87.864078
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1017,582,379,56,57.227139,37.266470,5.506391,160,422,31,165,106,42,17,122,105,14,3,44,23,21,17,1,55,15.732547,41.494592,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.294985,4.326450,2.261554,2.064897,1.671583,0.098328,5.408063,80,41,896,7.866273,4.031465,88.102262
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken

# 9. Create dominant field

In [21]:
def determine_dominant_group(row):
    age_columns = [
       'one_person_perc',
       'single_family_perc',  
       'other_household_types_perc',
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_household_composition_lsoa2021_gdb_df['dominant_household_composition_group'] = census2021_household_composition_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_household_composition_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_household_composition_pop_count,one_person_count,single_family_count,other_household_types_count,one_person_perc,single_family_perc,other_household_types_perc,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc,dominant_household_composition_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,841,373,417,51,44.351962,49.583829,6.064209,129,244,82,192,111,68,13,112,99,13,0,17,10,7,14,1,50,15.338882,29.013080,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,0.118906,5.945303,92,20,729,10.939358,2.378121,86.682521,single_family
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,824,409,369,46,49.635922,44.781553,5.582524,101,308,57,162,104,46,12,107,93,11,3,25,13,12,18,3,43,12.257282,37.378641,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,0.364078,5.218447,73,27,724,8.859223,3.276699,87.864078,"one_person, single_family"
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1017,582,379,56,57.227139,37.266470,5.506391,160,422,31,165,106,42,17,122,105,14,3,44,23,21,17,1,55,15.732547,41.494592,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.294985,4.326450,2.261554,2.064897,1.671583,0.098328,5.408063,80,41,896,7.866273,4.031465,88.102262,one_person
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 5

In [22]:
def determine_dominant_group(row):
    age_columns = [
       'one_person_aged_66_years_and_over_perc', 
       'one_person_other_perc',       
       'single_family_all_aged_66_years_and_over_perc',
       'single_family_married_or_civil_partnership_perc',
       'single_family_married_or_civil_partnership_no_children_perc',
       'single_family_married_or_civil_partnership_dependant_children_perc',
       'single_family_married_or_civil_partnership_nondep_children_perc',
       'single_family_cohabiting_perc',
       'single_family_cohabiting_no_children_perc',
       'single_family_cohabiting_dependant_children_perc',
       'single_family_cohabiting_nondep_children_perc',
       'single_family_lone_parent_perc',
       'single_family_lone_parent_dependant_children_perc',
       'single_family_lone_parent_nondep_children_perc',
       'single_family_other_perc',        
       'other_types_dependant_children_perc',
       'other_types_all_ft_students_and_all_aged_66_years_over_perc',
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value/2)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_household_composition_lsoa2021_gdb_df['dominant_detailed_household_composition_group'] = census2021_household_composition_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_household_composition_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_household_composition_pop_count,one_person_count,single_family_count,other_household_types_count,one_person_perc,single_family_perc,other_household_types_perc,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc,dominant_household_composition_group,dominant_detailed_household_composition_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,841,373,417,51,44.351962,49.583829,6.064209,129,244,82,192,111,68,13,112,99,13,0,17,10,7,14,1,50,15.338882,29.013080,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,0.118906,5.945303,92,20,729,10.939358,2.378121,86.682521,single_family,one_person_other
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,824,409,369,46,49.635922,44.781553,5.582524,101,308,57,162,104,46,12,107,93,11,3,25,13,12,18,3,43,12.257282,37.378641,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,0.364078,5.218447,73,27,724,8.859223,3.276699,87.864078,"one_person, single_family",one_person_other
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1017,582,379,56,57.227139,37.266470,5.506391,160,422,31,165,106,42,17,122,105,14,3,44,23,21,17,1,55,15.732547,41.494592,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.294985,4.326450,2.261554,2.064897,1.671583,0.098328,5.408063,80,41,896,7.866273,4.031465,88.102262,

In [23]:
def determine_dominant_group(row):
    age_columns = [
       'households_with_dependant_children_perc',
       'households_with_nondep_children_perc',
       'households_with_no_children_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value/2)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_household_composition_lsoa2021_gdb_df['dominant_households_by_children_group'] = census2021_household_composition_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_household_composition_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_household_composition_pop_count,one_person_count,single_family_count,other_household_types_count,one_person_perc,single_family_perc,other_household_types_perc,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc,dominant_household_composition_group,dominant_detailed_household_composition_group,dominant_households_by_children_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,841,373,417,51,44.351962,49.583829,6.064209,129,244,82,192,111,68,13,112,99,13,0,17,10,7,14,1,50,15.338882,29.013080,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,0.118906,5.945303,92,20,729,10.939358,2.378121,86.682521,single_family,one_person_other,households_with_no_children
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,824,409,369,46,49.635922,44.781553,5.582524,101,308,57,162,104,46,12,107,93,11,3,25,13,12,18,3,43,12.257282,37.378641,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,0.364078,5.218447,73,27,724,8.859223,3.276699,87.864078,"one_person, single_family",one_person_other,households_with_no_children
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1017,582,379,56,57.227139,37.266470,5.506391,160,422,31,165,106,42,17,122,105,14,3,44,23,21,17,1,55,15.732547,41.494592,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.29498

In [24]:
census2021_household_composition_lsoa2021_gdb_df.drop(['total_household_composition_pop_count'],1,inplace = True)
census2021_household_composition_lsoa2021_gdb_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14320\3426685100.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  census2021_household_composition_lsoa2021_gdb_df.drop(['total_household_composition_pop_count'],1,inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,one_person_count,single_family_count,other_household_types_count,one_person_perc,single_family_perc,other_household_types_perc,one_person_aged_66_years_and_over_count,one_person_other_count,single_family_all_aged_66_years_and_over_count,single_family_married_or_civil_partnership_count,single_family_married_or_civil_partnership_no_children_count,single_family_married_or_civil_partnership_dependant_children_count,single_family_married_or_civil_partnership_nondep_children_count,single_family_cohabiting_count,single_family_cohabiting_no_children_count,single_family_cohabiting_dependant_children_count,single_family_cohabiting_nondep_children_count,single_family_lone_parent_count,single_family_lone_parent_dependant_children_count,single_family_lone_parent_nondep_children_count,single_family_other_count,other_types_dependant_children_count,other_types_all_ft_students_and_all_aged_66_years_over_count,one_person_aged_66_years_and_over_perc,one_person_other_perc,single_family_all_aged_66_years_and_over_perc,single_family_married_or_civil_partnership_perc,single_family_married_or_civil_partnership_no_children_perc,single_family_married_or_civil_partnership_dependant_children_perc,single_family_married_or_civil_partnership_nondep_children_perc,single_family_cohabiting_perc,single_family_cohabiting_no_children_perc,single_family_cohabiting_dependant_children_perc,single_family_cohabiting_nondep_children_perc,single_family_lone_parent_perc,single_family_lone_parent_dependant_children_perc,single_family_lone_parent_nondep_children_perc,single_family_other_perc,other_types_dependant_children_perc,other_types_all_ft_students_and_all_aged_66_years_over_perc,households_with_dependant_children_count,households_with_nondep_children_count,households_with_no_children_count,households_with_dependant_children_perc,households_with_nondep_children_perc,households_with_no_children_perc,dominant_household_composition_group,dominant_detailed_household_composition_group,dominant_households_by_children_group
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,373,417,51,44.351962,49.583829,6.064209,129,244,82,192,111,68,13,112,99,13,0,17,10,7,14,1,50,15.338882,29.013080,9.750297,22.829964,13.198573,8.085612,1.545779,13.317479,11.771700,1.545779,0.000000,2.021403,1.189061,0.832342,1.664685,0.118906,5.945303,92,20,729,10.939358,2.378121,86.682521,single_family,one_person_other,households_with_no_children
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,409,369,46,49.635922,44.781553,5.582524,101,308,57,162,104,46,12,107,93,11,3,25,13,12,18,3,43,12.257282,37.378641,6.917476,19.660194,12.621359,5.582524,1.456311,12.985437,11.286408,1.334951,0.364078,3.033981,1.577670,1.456311,2.184466,0.364078,5.218447,73,27,724,8.859223,3.276699,87.864078,"one_person, single_family",one_person_other,households_with_no_children
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,582,379,56,57.227139,37.266470,5.506391,160,422,31,165,106,42,17,122,105,14,3,44,23,21,17,1,55,15.732547,41.494592,3.048181,16.224189,10.422812,4.129794,1.671583,11.996067,10.324484,1.376598,0.294985,4.326450,2.261554,2.064897,1.671583,0.098328,5.40

In [25]:
census2021_household_composition_lsoa2021_gdb_df.columns

Index(['FID', 'lsoa21cd', 'lsoa21nm', 'geometry', 'msoa21cd', 'msoa21nm',
       'lad22cd', 'lad22nm', 'wd22cd', 'wd22nm', 'rgn22cd', 'rgn22nm',
       'data_source', 'data_resolution', 'data_time_period', 'data_web_link',
       'area_ha', 'one_person_count', 'single_family_count',
       'other_household_types_count', 'one_person_perc', 'single_family_perc',
       'other_household_types_perc', 'one_person_aged_66_years_and_over_count',
       'one_person_other_count',
       'single_family_all_aged_66_years_and_over_count',
       'single_family_married_or_civil_partnership_count',
       'single_family_married_or_civil_partnership_no_children_count',
       'single_family_married_or_civil_partnership_dependant_children_count',
       'single_family_married_or_civil_partnership_nondep_children_count',
       'single_family_cohabiting_count',
       'single_family_cohabiting_no_children_count',
       'single_family_cohabiting_dependant_children_count',
       'single_family_cohabiti

# 9. Upload to geodatabase

In [26]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_household_composition"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [27]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_household_composition_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_household_composition_lsoa2021_gdb_dff.set_crs(epsg=27700, inplace=True)

In [28]:
# Publish the GeoDataFrame to PostGIS
census2021_household_composition_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_household_composition
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_household_composition
